In [1]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp
from queue import Queue
from threading import Thread
from tqdm import tqdm
import math
import time

# -------------------------
# CONFIGURATION
# -------------------------

TILE_SIZE = 512
BATCH_SIZE = 32
QUEUE_SIZE = 128

tile_queue = Queue(maxsize=QUEUE_SIZE)
result_queue = Queue(maxsize=QUEUE_SIZE)

# -------------------------
# GPU Batch Processing
# -------------------------

def process_batch(batch_windows, batch_red, batch_nir):

    red_stack = np.asarray(batch_red, dtype="float32")
    nir_stack = np.asarray(batch_nir, dtype="float32")

    red_gpu = cp.asarray(red_stack)
    nir_gpu = cp.asarray(nir_stack)

    ndvi_gpu = (nir_gpu - red_gpu) / (nir_gpu + red_gpu + 1e-6)

    ndvi_stack = cp.asnumpy(ndvi_gpu)

    for i in range(len(batch_windows)):
        result_queue.put((batch_windows[i], ndvi_stack[i]))


# -------------------------
# READER THREAD
# -------------------------

def reader(red_path, nir_path):

    with rasterio.open(red_path) as red_src, rasterio.open(nir_path) as nir_src:

        h, w = red_src.height, red_src.width

        for r in range(0, h, TILE_SIZE):
            for c in range(0, w, TILE_SIZE):

                win = Window(c, r, min(TILE_SIZE, w-c), min(TILE_SIZE, h-r))

                red = red_src.read(
                    1,
                    window=win,
                    boundless=True,
                    fill_value=0,
                    out_shape=(TILE_SIZE, TILE_SIZE)
                )

                nir = nir_src.read(
                    1,
                    window=win,
                    boundless=True,
                    fill_value=0,
                    out_shape=(TILE_SIZE, TILE_SIZE)
                )

                tile_queue.put((win, red, nir))

    tile_queue.put(None)


# -------------------------
# WORKER THREAD
# -------------------------

def worker():

    batch_windows = []
    batch_red = []
    batch_nir = []

    while True:

        item = tile_queue.get()

        if item is None:

            if batch_windows:
                process_batch(batch_windows, batch_red, batch_nir)

            result_queue.put(None)
            break

        win, red, nir = item

        batch_windows.append(win)
        batch_red.append(red)
        batch_nir.append(nir)

        if len(batch_windows) == BATCH_SIZE:

            process_batch(batch_windows, batch_red, batch_nir)

            batch_windows = []
            batch_red = []
            batch_nir = []


# -------------------------
# WRITER THREAD
# -------------------------

def writer(output_path, profile, total_tiles):

    with rasterio.open(output_path, "w", **profile) as dst:

        pbar = tqdm(total=total_tiles, desc="Processing Tiles")

        while True:

            item = result_queue.get()

            if item is None:
                break

            win, ndvi = item

            dst.write(
                ndvi[:win.height, :win.width],
                1,
                window=win
            )

            pbar.update(1)

        pbar.close()


# -------------------------
# MAIN
# -------------------------

def main():

    red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m_converted.tif"
    nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m_converted.tif"

    output_path = "SOGNA_pipeline_output.tif"

    start = time.time()

    with rasterio.open(red_path) as src:

        h, w = src.height, src.width

        profile = src.profile

        profile.update(
            dtype="float32",
            count=1,
            driver="GTiff",
            tiled=True,
            compress="lzw"
        )

        total_tiles = math.ceil(h / TILE_SIZE) * math.ceil(w / TILE_SIZE)

    reader_thread = Thread(target=reader, args=(red_path, nir_path))
    worker_thread = Thread(target=worker)
    writer_thread = Thread(target=writer, args=(output_path, profile, total_tiles))

    reader_thread.start()
    worker_thread.start()
    writer_thread.start()

    reader_thread.join()
    worker_thread.join()
    writer_thread.join()

    print(f"\nPipeline finished in {time.time() - start:.2f} seconds")


if __name__ == "__main__":
    main()

Processing Tiles: 100%|██████████| 484/484 [00:05<00:00, 85.24it/s] 



Pipeline finished in 7.61 seconds
